In [ ]:
"""
=============================================================
  GREY-BOX MODEL  —  VESSEL VERSION v7.5  (EXPLICIT 1s CADENCE + PER-FILE PURGE)
  Author : fansuri

  CHANGES vs v7.4:
  ─────────────────────────────────────────────────────────
  1. resample_seconds: 60 → 1. The bridge (v10.1) matches whatever
     cadence lives in meta.pkl, and the ESP32/GPS stream reports at
     roughly 1Hz, not once/minute. Training at resample_seconds=60
     meant the model's lag features were calibrated to a cadence the
     live system would never actually deliver.

     IMPORTANT — what "resample_seconds=1" does and doesn't do:
     _resample_to_uniform_cadence() bins each file onto the target
     grid via mean-aggregation, then DROPS empty bins. It never
     forward-fills or interpolates. So for the high-rate road-test
     CSVs (near-1Hz already), resampling to 1s is basically a no-op
     denoise. For the Jati Six Excel logs (~60s native spacing),
     resampling to a 1s grid does NOT fabricate 60 fake 1-second
     samples per real one — the empty bins just get dropped, and
     the file keeps reporting at its own real ~60s cadence. No
     synthetic data is created in either domain. The tradeoff is
     that SOG_LAG1 etc. then mean genuinely different real-world
     time gaps depending on which file a row came from (~1s for
     road tests, ~60s for Jati Six) — this is the explicit,
     accepted tradeoff, not a bug.

  2. forecast_horizon_s: 60 → 30. Restored to the original 30s
     alert horizon now that the high-rate files can actually hit it
     exactly. The Jati Six files still can't (~60s native cadence),
     and will report their nearest achievable H — expected, not an
     error (see the softened warning text in _load_one_csv below).

  3. PER-FILE PURGE GAP FIX. train() previously converted
     purge_gap_seconds → rows using ONE global cadence assumption
     (CFG["resample_seconds"]) applied to every file. That was fine
     back when resample_seconds=60 roughly matched the Jati Six
     files, but with resample_seconds=1 it would compute purge as
     if every file were 1s apart — inflating the Jati Six purge
     window by ~60x and silently starving those files of train/val
     rows (the exact bug class the v7.4 docstring claimed to have
     fixed). Fixed by computing purge per file, from that file's own
     actual median timestamp gap (stored as df["_DT_MEDIAN_S"]),
     computed once in _load_one_csv and reused for both the horizon
     shift and the purge conversion.

  Everything else — Excel support, file-based test holdout via
  "split", alpha residual-shrinkage, physics prior, PINN residual,
  XGBoost stack, persistence baseline — is UNCHANGED from v7.4.
=============================================================
"""

import os
import shutil
import warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")

# ─────────────────────────── CONSTANTS ────────────────────────
KN_TO_KMH = 1.852
KMH_TO_KN = 1.0 / KN_TO_KMH

MAX_SPEED_KN   = 35.0
MAX_WIND_KN    = 80.0
MAX_WAVE_M     = 3.0
MIN_SOG_KN     = 0.05

# ─────────────────────────── CONFIG ───────────────────────────
CFG = {
    "checkpoint_dir" : "checkpoints_fyp(vessel_forecast)",

    "data_files": [

        # ── real vessel operating data (Jati Six) ──────────────
        # columns used: VESSEL_KNOTS, SIG_WAVE_HEIGHT_M, WIND_SPEED_KNOTS
        # SIG_WAVE_HEIGHT_M is already in metres → wave_unit="m"
        # samples ~once/minute — this stays at native ~60s cadence
        # even after the global resample_seconds=1 setting below,
        # since resampling only bins/denoises, it never fabricates
        # samples for a file that doesn't have them (see header note).
        {
            "path"      : r"C:\Users\fansuri\Documents\FPY\Jati Six - April 25.xlsx",
            "wave_unit" : "m",
            "split"     : "train",
        },
        {
            "path"      : r"C:\Users\fansuri\Documents\FPY\Jati Six - May 25 (version 1).xlsx",
            "wave_unit" : "m",
            "split"     : "train",
        },
        {
            "path"      : r"C:\Users\fansuri\Documents\FPY\Jati Six - June 25.xlsx",
            "wave_unit" : "m",
            "split"     : "train",
        },
        {
            "path"      : r"C:\Users\fansuri\Documents\FPY\Jati Six - July 2025 (1).xlsx",
            "wave_unit" : "m",
            "split"     : "test",
        },
    ],

    # ── UNIFORM CADENCE (bin + denoise, never interpolate) ──────
    # Every file is resampled (mean aggregation) onto this cadence
    # before anything else happens. Set to 1 to match the live
    # ESP32/GPS stream's roughly-1Hz reporting rate — this is what
    # the deployed bridge actually delivers, so lag features learnt
    # here should be calibrated to real seconds-apart, not minutes.
    # Files with sparser native sampling (e.g. the Jati Six Excel
    # logs, ~60s apart) simply keep their own real cadence — empty
    # bins get dropped, never filled. Set to 0 to disable entirely.
    "resample_seconds"   : 60,

    # ── FORECASTING ────────────────────────────────────────────
    # 30s ahead. Exactly achievable for the high-rate road-test
    # files at resample_seconds=1. The Jati Six files (~60s native
    # cadence) can't hit 30s exactly and will fall back to their
    # nearest achievable H — see the per-file warning text below;
    # that's expected for those files, not an error.
    "forecast_horizon_s" : 60,

    "min_sog_kn"         : MIN_SOG_KN,
    "max_wave_m"         : MAX_WAVE_M,
    "label_smooth_k"     : 3,

    "lr"                 : 1e-3,
    "epochs"             : 300,
    "patience"           : 40,
    "batch_size"         : 2048,
    "lambda_mono"        : 0.5,
    "hidden_dim"         : 32,
    "num_layers"         : 2,
    "dropout"            : 0.2,
    "weight_decay"       : 2e-3,
    "mono_sample_size"   : 512,

    "xgb_n_estimators"   : 2000,
    "xgb_max_depth"      : 3,
    "xgb_lr"             : 0.01,
    "xgb_subsample"      : 0.7,
    "xgb_colsample"      : 0.7,
    "xgb_min_child"      : 50,
    "xgb_early_stopping" : 50,
    "xgb_reg_alpha"      : 1.0,
    "xgb_reg_lambda"     : 5.0,

    "train_ratio"        : 0.75,
    "val_ratio"          : 0.10,
    # purge gap, in SECONDS. Converted to rows PER FILE at runtime
    # using each file's own actual median timestamp gap (see
    # df["_DT_MEDIAN_S"] in _load_one_csv) — NOT one global cadence
    # assumption. 600s = 10 minutes of purge around each val slice.
    "purge_gap_seconds"  : 600,
}

os.makedirs(CFG["checkpoint_dir"], exist_ok=True)
PINN_PATH    = os.path.join(CFG["checkpoint_dir"], "best_pinn.pt")
XGB_PATH     = os.path.join(CFG["checkpoint_dir"], "xgb.json")
SCALER_PATH  = os.path.join(CFG["checkpoint_dir"], "scaler.pkl")
PHYSICS_PATH = os.path.join(CFG["checkpoint_dir"], "physics_coef.pkl")
META_PATH    = os.path.join(CFG["checkpoint_dir"], "meta.pkl")
PLOT_PATH    = os.path.join(CFG["checkpoint_dir"], "results_vessel_knots.png")

FEATURE_COLS = [
    "TRUE_WIND_KN",
    "WAVE_HS_M",
    "SOG_LAG1",
    "SOG_LAG2",
    "SOG_LAG3",
    "SOG_ROLL5",
    "SOG_STD5",
    "SOG_DIFF1",
    "SOG_DIFF3",
    "HOUR_SIN",
    "HOUR_COS",
    "MONTH_SIN",
    "MONTH_COS",
]
TARGET = "SOG_KN_FUTURE"

COLORS = {
    "physics": "#e07b54",
    "pinn"   : "#5b8dd9",
    "xgb"    : "#57a773",
    "ideal"  : "#aaaaaa",
    "persist": "#9b59b6",
}


# ══════════════════════════════════════════════════════════════
#  HELPER — find a column by keyword (case-insensitive)
# ══════════════════════════════════════════════════════════════
def _find_col(df: pd.DataFrame, *keywords: str, exclude=frozenset()) -> str:
    cols_lower = {c.lower(): c for c in df.columns if c not in exclude}
    for kw in keywords:
        for lower, original in cols_lower.items():
            if kw.lower() in lower:
                return original
    raise KeyError(
        f"Could not find a column matching {keywords} in: "
        f"{[c for c in df.columns if c not in exclude]}\n"
        f"Please check your file's headers and update CFG['data_files'] if needed."
    )


# ══════════════════════════════════════════════════════════════
#  UNIFORM CADENCE RESAMPLING
#  Bins every file onto the same target grid via mean aggregation,
#  then DROPS empty bins. This denoises high-rate files (GPS/sensor
#  jitter averaged within each bin) but does NOT fabricate data for
#  sparse files — a file with real samples every ~60s will simply
#  keep reporting every ~60s even if the target grid is 1s, because
#  there's nothing to put in the 59 empty bins in between.
# ══════════════════════════════════════════════════════════════
def _resample_to_uniform_cadence(df: pd.DataFrame, seconds: int) -> pd.DataFrame:
    if seconds <= 0:
        return df
    d = df.set_index("timestamp")[["wind_kn", "sog_kn", "wave_hs_m"]]
    agg = d.resample(f"{seconds}s").mean()
    agg = agg.dropna(subset=["sog_kn"]).reset_index()
    return agg


# ══════════════════════════════════════════════════════════════
#  LAG FEATURES  (unchanged — built from CURRENT SOG only)
# ══════════════════════════════════════════════════════════════
def add_lag_features(df: pd.DataFrame,
                     seed_sog: pd.Series = None) -> pd.DataFrame:
    df = df.copy()
    s  = df["SOG_KN"].copy()   # CURRENT sog — never the future target

    if seed_sog is not None:
        seed  = seed_sog.iloc[-10:].reset_index(drop=True)
        s_ext = pd.concat([seed, s], ignore_index=True)
        off   = len(seed)
    else:
        s_ext = s
        off   = 0

    df["SOG_LAG1"]  = s_ext.shift(1).bfill().iloc[off:].values
    df["SOG_LAG2"]  = s_ext.shift(2).bfill().iloc[off:].values
    df["SOG_LAG3"]  = s_ext.rolling(3, min_periods=1).mean().bfill().iloc[off:].values
    df["SOG_ROLL5"] = s_ext.rolling(5, min_periods=1).mean().bfill().iloc[off:].values
    df["SOG_STD5"]  = (s_ext.rolling(5, min_periods=1).std()
                            .fillna(0.0).iloc[off:].values)
    df["SOG_DIFF1"] = s_ext.diff(1).bfill().iloc[off:].values
    df["SOG_DIFF3"] = s_ext.diff(3).bfill().iloc[off:].values
    return df


# ══════════════════════════════════════════════════════════════
#  SINGLE-FILE LOADER
#  Resamples to the target cadence BEFORE label smoothing or the
#  horizon shift. Also computes and stores this file's ACTUAL
#  median timestamp gap (_DT_MEDIAN_S) once, so train() can size
#  the purge gap per file instead of assuming one global cadence.
# ══════════════════════════════════════════════════════════════
def _load_one_csv(path: str, wave_unit: str = "m") -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Data file not found: {path}\n"
            f"Make sure the file is in the correct folder."
        )

    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(path, sheet_name=0)
    else:
        df = pd.read_csv(path)

    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    print(f"    {path}: {len(df):,} raw rows")
    print(f"    Columns detected: {df.columns.tolist()}")

    ts_col   = _find_col(df, "timestamp", "time", "date", "datetime", "utc")
    wind_col = _find_col(df, "wind_kn", "wind_speed", "wind",
                         exclude={ts_col})
    sog_col  = _find_col(df, "sog_kn", "sog", "vessel_knots", "speed",
                         exclude={ts_col, wind_col})
    wave_col = _find_col(df, "wave_hs", "wave_height", "hs", "wave",
                         exclude={ts_col, wind_col, sog_col})

    print(f"    Mapped → timestamp='{ts_col}'  wind='{wind_col}'  "
          f"sog='{sog_col}'  wave='{wave_col}'")

    df["timestamp"] = pd.to_datetime(df[ts_col], errors="coerce")
    df = (df.dropna(subset=["timestamp"])
            .sort_values("timestamp")
            .reset_index(drop=True))

    df["wind_kn"] = pd.to_numeric(df[wind_col], errors="coerce").clip(0, MAX_WIND_KN)
    df["sog_kn"]  = pd.to_numeric(df[sog_col],  errors="coerce").clip(0, MAX_SPEED_KN)

    raw_wave = pd.to_numeric(df[wave_col], errors="coerce")
    if wave_unit == "cm":
        wave_m = raw_wave / 100.0
        print(f"    wave_unit=cm → converted to metres  "
              f"(raw range {raw_wave.min():.1f}–{raw_wave.max():.1f} cm  "
              f"→  {wave_m.min():.4f}–{wave_m.max():.4f} m)")
    else:
        wave_m = raw_wave
        print(f"    wave_unit=m  → kept as metres  "
              f"(range {wave_m.min():.4f}–{wave_m.max():.4f} m)")
    df["wave_hs_m"] = wave_m.clip(0, MAX_WAVE_M)

    df = df.ffill().bfill()

    # ── UNIFORM CADENCE RESAMPLE ────────────────────────────────
    resample_s = CFG.get("resample_seconds", 0)
    if resample_s > 0:
        n_before = len(df)
        df = _resample_to_uniform_cadence(
            df[["timestamp", "wind_kn", "sog_kn", "wave_hs_m"]], resample_s
        )
        df = df.ffill().bfill()
        print(f"    Resampled onto {resample_s}s grid (empty bins dropped, "
              f"not filled): {n_before:,} rows → {len(df):,} rows")

    # ── this file's ACTUAL cadence, post-resample — used for both
    #    the forecast horizon below AND per-file purge sizing in
    #    train(). Computed once here so both stay consistent. ────
    dt_series      = df["timestamp"].diff().dt.total_seconds()
    dt_median_file = dt_series.median()
    if not np.isfinite(dt_median_file) or dt_median_file <= 0:
        dt_median_file = 1.0
    df["_DT_MEDIAN_S"] = dt_median_file

    df["SOG_KN"]       = df["sog_kn"]
    df["WIND_KN"]      = df["wind_kn"]
    df["WAVE_HS_M"]    = df["wave_hs_m"]
    df["TRUE_WIND_KN"] = np.maximum(0.0, df["WIND_KN"] - df["SOG_KN"])

    k = CFG["label_smooth_k"]
    if k > 1:
        df["SOG_KN"] = (df["SOG_KN"]
                        .rolling(k, min_periods=1, center=True)
                        .mean())

    # ── FORECAST TARGET — shift SOG forward in time, per file ──
    horizon_s = CFG.get("forecast_horizon_s", 0)
    if horizon_s > 0:
        dt_median = dt_median_file
        H = max(1, round(horizon_s / dt_median))
        actual_horizon_s = H * dt_median
        df["SOG_KN_FUTURE"] = df["SOG_KN"].shift(-H)
        n_before = len(df)
        df = df.dropna(subset=["SOG_KN_FUTURE"]).reset_index(drop=True)

        if abs(actual_horizon_s - horizon_s) > 0.5:
            print(f"    ⚠  REQUESTED {horizon_s}s horizon but this file's NATIVE "
                  f"cadence is {dt_median:.1f}s — resample_seconds={resample_s}s only "
                  f"bins/denoises, it can't fabricate samples for a sparser file. "
                  f"Using H={H} row(s) = {actual_horizon_s:.0f}s ahead instead. "
                  f"Expected for low-frequency sources (e.g. the Jati Six Excel "
                  f"logs); if a high-rate road-test CSV shows this, check for a "
                  f"large gap in that file.")
        else:
            print(f"    Forecast horizon: {horizon_s}s → H={H} rows "
                  f"(dt≈{dt_median:.2f}s/sample) — dropped {n_before - len(df)} "
                  f"tail rows with no future target")
    else:
        df["SOG_KN_FUTURE"] = df["SOG_KN"]

    before = len(df)
    df = df[df["SOG_KN"] >= CFG["min_sog_kn"]].reset_index(drop=True)
    print(f"    Removed {before - len(df):,} near-stationary rows "
          f"→ {len(df):,} usable rows")

    hour  = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60.0
    month = df["timestamp"].dt.month
    df["HOUR_SIN"]  = np.sin(2 * np.pi * hour        / 24.0)
    df["HOUR_COS"]  = np.cos(2 * np.pi * hour        / 24.0)
    df["MONTH_SIN"] = np.sin(2 * np.pi * (month - 1) / 12.0)
    df["MONTH_COS"] = np.cos(2 * np.pi * (month - 1) / 12.0)

    return df


# ══════════════════════════════════════════════════════════════
#  DATA LOADING & CLEANING
# ══════════════════════════════════════════════════════════════
def load_and_clean() -> pd.DataFrame:
    horizon_s   = CFG.get("forecast_horizon_s", 0)
    resample_s  = CFG.get("resample_seconds", 0)
    print(f"  Loading {len(CFG['data_files'])} CSV/Excel file(s) …")
    if resample_s > 0:
        print(f"  CADENCE: all files binned onto a {resample_s}s grid "
              f"(empty bins dropped — sparse files keep their native rate)")
    if horizon_s > 0:
        print(f"  MODE: FORECASTING — target is SOG {horizon_s}s ahead of the inputs")
    else:
        print("  MODE: NOWCAST — target is SOG at the same timestamp as inputs")

    frames = []
    for file_cfg in CFG["data_files"]:
        f_df = _load_one_csv(
            path      = file_cfg["path"],
            wave_unit = file_cfg.get("wave_unit", "m"),
        )
        f_df["SPLIT_TAG"]   = file_cfg.get("split", "train")
        f_df["SOURCE_FILE"] = os.path.basename(file_cfg["path"])
        frames.append(f_df)

    df = pd.concat(frames, ignore_index=True)
    print(f"\n  Combined total: {len(df):,} rows "
          f"(from {len(CFG['data_files'])} files)")
    n_test_tag = int((df["SPLIT_TAG"] == "test").sum())
    print(f"  Rows tagged split='test' (held-out entirely): {n_test_tag:,}")

    print()
    print("  Data summary (all in knots / metres):")
    print(f"    Apparent wind   : {df['WIND_KN'].min():.2f}–{df['WIND_KN'].max():.2f} kn  "
          f"mean={df['WIND_KN'].mean():.2f}")
    print(f"    True wind       : {df['TRUE_WIND_KN'].min():.2f}–{df['TRUE_WIND_KN'].max():.2f} kn  "
          f"mean={df['TRUE_WIND_KN'].mean():.2f}  ← model input")
    print(f"    Wave height Hs  : {df['WAVE_HS_M'].min():.4f}–{df['WAVE_HS_M'].max():.4f} m  "
          f"mean={df['WAVE_HS_M'].mean():.4f}")
    print(f"    SOG (current)   : {df['SOG_KN'].min():.2f}–{df['SOG_KN'].max():.2f} kn  "
          f"mean={df['SOG_KN'].mean():.2f}  (smoothed per-file)")
    print(f"    SOG (target)    : {df[TARGET].min():.2f}–{df[TARGET].max():.2f} kn  "
          f"mean={df[TARGET].mean():.2f}  ← what the model is trained to hit")
    print(f"    SOG km/h equiv  : {df['SOG_KN'].min()*KN_TO_KMH:.1f}–"
          f"{df['SOG_KN'].max()*KN_TO_KMH:.1f} km/h")
    print()
    return df


# ══════════════════════════════════════════════════════════════
#  PHYSICS MODEL  (unchanged)
# ══════════════════════════════════════════════════════════════
def build_physics_features(wind, wave, lag1):
    ones = np.ones_like(wind)
    return np.column_stack([ones, wind, wave,
                             wind**2, wave**2, wind * wave,
                             lag1])


def calibrate_physics(wind_tr, wave_tr, lag1_tr, y_tr):
    Phi   = build_physics_features(wind_tr, wave_tr, lag1_tr)
    ridge = Ridge(alpha=10.0, fit_intercept=False)
    ridge.fit(Phi, y_tr)
    coef  = ridge.coef_.copy()
    print(f"  Physics coef: intercept={coef[0]:.4f}  wind={coef[1]:.4f}"
          f"  wave={coef[2]:.4f}  lag1={coef[6]:.4f}")
    print(f"                wind²={coef[3]:.5f}  wave²={coef[4]:.5f}"
          f"  wind·wave={coef[5]:.5f}")
    if coef[6] < 0.3:
        print(f"  ⚠  lag1 coef={coef[6]:.4f} is low — physics may still underfit")
    return coef


def physics_predict(wind, wave, lag1, coef):
    return np.clip(
        build_physics_features(wind, wave, lag1) @ coef,
        0.0, MAX_SPEED_KN
    )


# ══════════════════════════════════════════════════════════════
#  PINN  (unchanged)
# ══════════════════════════════════════════════════════════════
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
    def forward(self, x):
        return x + self.block(x)


class PINN(nn.Module):
    def __init__(self, in_dim, hidden=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResBlock(hidden, dropout) for _ in range(n_layers)]
        )
        self.head = nn.Linear(hidden, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.res_blocks(self.input_proj(x)))


def _save_torch(state_dict, path):
    tmp = path + ".tmp"
    torch.save(state_dict, tmp)
    shutil.move(tmp, path)


def _save_joblib(obj, path):
    tmp = path + ".tmp"
    joblib.dump(obj, tmp)
    shutil.move(tmp, path)


# ══════════════════════════════════════════════════════════════
#  METRICS  (unchanged)
# ══════════════════════════════════════════════════════════════
def metrics_kn(y_true_kn, y_pred_kn):
    ae = np.abs(y_true_kn - y_pred_kn)
    return {
        "rmse_kn"   : float(np.sqrt(mean_squared_error(y_true_kn, y_pred_kn))),
        "mae_kn"    : float(mean_absolute_error(y_true_kn, y_pred_kn)),
        "r2"        : float(r2_score(y_true_kn, y_pred_kn)),
        "p0_5_kn"   : float(np.mean(ae <= 0.5)  * 100),
        "p1_kn"     : float(np.mean(ae <= 1.0)  * 100),
        "p2_kn"     : float(np.mean(ae <= 2.0)  * 100),
        "p5_kn"     : float(np.mean(ae <= 5.0)  * 100),
        "rmse_kmh"  : float(np.sqrt(mean_squared_error(
                          y_true_kn * KN_TO_KMH, y_pred_kn * KN_TO_KMH))),
        "mae_kmh"   : float(mean_absolute_error(
                          y_true_kn * KN_TO_KMH, y_pred_kn * KN_TO_KMH)),
        "p1_kmh"    : float(np.mean(ae * KN_TO_KMH <= 1.0) * 100),
        "p2_kmh"    : float(np.mean(ae * KN_TO_KMH <= 2.0) * 100),
    }


# ══════════════════════════════════════════════════════════════
#  TRAINING
# ══════════════════════════════════════════════════════════════
def train():
    horizon_s  = CFG.get("forecast_horizon_s", 0)
    resample_s = CFG.get("resample_seconds", 0)
    mode_label = f"FORECAST +{horizon_s}s" if horizon_s > 0 else "NOWCAST"

    print("\n" + "═"*72)
    print(f"  GREY-BOX MODEL — VESSEL v7.5  ({mode_label})")
    print("  Physics: wind+wave+lag1 prior | PINN residual (α-shrunk) | XGBoost stack")
    print(f"  13 features | target grid {resample_s}s | per-file purge sizing | file-based test holdout")
    print("═"*72)

    df = load_and_clean()

    has_file_test = bool((df["SPLIT_TAG"] == "test").any())
    purge_gap_s = CFG.get("purge_gap_seconds", 0)
    print(f"  Purge gap target: {purge_gap_s}s — converted to rows PER FILE below, "
          f"using each file's own actual cadence (not one global assumption)")

    if has_file_test:
        te_raw = (df[df["SPLIT_TAG"] == "test"]
                  .sort_values("timestamp").reset_index(drop=True))

        # ── PER-FILE train/val split, with PER-FILE purge sizing ─
        pool_names = df.loc[df["SPLIT_TAG"] == "train", "SOURCE_FILE"].unique()
        va_frac = CFG["val_ratio"] / (CFG["train_ratio"] + CFG["val_ratio"])

        print(f"\n  Per-file train/val split:")
        tr_pieces, va_pieces = [], []
        for name in pool_names:
            fdf = (df[(df["SPLIT_TAG"] == "train") & (df["SOURCE_FILE"] == name)]
                   .sort_values("timestamp").reset_index(drop=True))
            n_f = len(fdf)
            file_cadence_s = float(fdf["_DT_MEDIAN_S"].iloc[0]) if n_f else 1.0
            purge = (max(1, int(purge_gap_s // file_cadence_s))
                     if purge_gap_s > 0 else 0)
            n_va_f = int(n_f * va_frac)
            n_tr_f = n_f - n_va_f - purge
            if n_tr_f <= 0 or n_va_f <= 0:
                print(f"    ⚠  '{name}' too small ({n_f} rows, cadence≈{file_cadence_s:.0f}s, "
                      f"purge={purge} rows) for this split — using it for train only, no val slice")
                tr_pieces.append(fdf)
                continue
            tr_pieces.append(fdf.iloc[:n_tr_f])
            va_pieces.append(fdf.iloc[n_tr_f + purge : n_tr_f + purge + n_va_f])
            print(f"    '{name}': {n_f:,} rows  cadence≈{file_cadence_s:.0f}s  "
                  f"purge={purge} rows → train={n_tr_f:,}  val={n_va_f:,}")

        tr_raw = (pd.concat(tr_pieces, ignore_index=True) if tr_pieces
                  else df.iloc[0:0].copy())
        va_raw = (pd.concat(va_pieces, ignore_index=True) if va_pieces
                  else df.iloc[0:0].copy())

        print(f"\n  File-based split → test file(s) held out entirely: "
              f"{len(te_raw):,} rows")
        print(f"  Train/val pool (per-file split, combined): "
              f"train={len(tr_raw):,}  val={len(va_raw):,}")
        test_seed = None
    else:
        n      = len(df)
        n_tr   = int(n * CFG["train_ratio"])
        n_va   = int(n * CFG["val_ratio"])
        df_sorted = df.sort_values("timestamp").reset_index(drop=True)
        cadence_s = float(df_sorted["_DT_MEDIAN_S"].median()) \
                    if "_DT_MEDIAN_S" in df_sorted.columns else 1.0
        purge = (max(1, int(purge_gap_s // cadence_s))
                 if purge_gap_s > 0 else 0)
        print(f"  No file tagged split='test' — falling back to chronological "
              f"ratio split (cadence≈{cadence_s:.0f}s → purge={purge} rows)")
        tr_raw = df_sorted.iloc[:n_tr].copy()
        va_raw = df_sorted.iloc[n_tr + purge : n_tr + purge + n_va].copy()
        te_raw = df_sorted.iloc[n_tr + purge + n_va + purge:].copy()
        test_seed = pd.concat([tr_raw["SOG_KN"], va_raw["SOG_KN"]],
                              ignore_index=True)

    tr_df = add_lag_features(tr_raw, seed_sog=None)
    va_df = add_lag_features(va_raw, seed_sog=tr_raw["SOG_KN"] if len(tr_raw) else None)
    te_df = add_lag_features(te_raw, seed_sog=test_seed)

    print(f"\n  Final split → train:{len(tr_df):,}  val:{len(va_df):,}  test:{len(te_df):,}")

    Xtr = tr_df[FEATURE_COLS].values.astype(np.float32)
    ytr = tr_df[TARGET].values.astype(np.float32)
    Xva = va_df[FEATURE_COLS].values.astype(np.float32)
    yva = va_df[TARGET].values.astype(np.float32)
    Xte = te_df[FEATURE_COLS].values.astype(np.float32)
    yte = te_df[TARGET].values.astype(np.float32)

    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(Xtr)
    Xva_s  = scaler.transform(Xva)
    Xte_s  = scaler.transform(Xte)
    _save_joblib(scaler, SCALER_PATH)
    print(f"  Scaler saved — {scaler.n_features_in_} features")

    wtr  = tr_df["TRUE_WIND_KN"].values.astype(np.float32)
    vtr  = tr_df["WAVE_HS_M"].values.astype(np.float32)
    l1tr = tr_df["SOG_LAG1"].values.astype(np.float32)
    wva  = va_df["TRUE_WIND_KN"].values.astype(np.float32)
    vva  = va_df["WAVE_HS_M"].values.astype(np.float32)
    l1va = va_df["SOG_LAG1"].values.astype(np.float32)
    wte  = te_df["TRUE_WIND_KN"].values.astype(np.float32)
    vte  = te_df["WAVE_HS_M"].values.astype(np.float32)
    l1te = te_df["SOG_LAG1"].values.astype(np.float32)

    print("\n  Calibrating physics prior …")
    coef = calibrate_physics(wtr, vtr, l1tr, ytr)
    _save_joblib(coef, PHYSICS_PATH)

    Vp_tr = physics_predict(wtr, vtr, l1tr, coef)
    Vp_va = physics_predict(wva, vva, l1va, coef)
    Vp_te = physics_predict(wte, vte, l1te, coef)

    ph_r2_tr = r2_score(ytr, Vp_tr)
    ph_r2_te = r2_score(yte, Vp_te)
    print(f"  Physics R²: train={ph_r2_tr:.4f}  test={ph_r2_te:.4f}")
    if ph_r2_te < 0:
        print("  ⚠  Physics R² negative on test — PINN+XGB will correct this.")

    res_tr   = ytr - Vp_tr
    res_mean = float(res_tr.mean())
    res_std  = float(res_tr.std()) + 1e-6
    CLIP_SIG = 3
    res_clip = np.clip(res_tr,
                       res_mean - CLIP_SIG * res_std,
                       res_mean + CLIP_SIG * res_std)
    n_clip   = int(np.sum(np.abs(res_tr - res_mean) > CLIP_SIG * res_std))
    res_va   = yva - Vp_va
    res_tr_n = (res_clip - res_mean) / res_std
    res_va_n = (res_va   - res_mean) / res_std

    print(f"  Residual — mean:{res_tr.mean():.4f}  std:{res_tr.std():.4f}  "
          f"min:{res_tr.min():.4f}  max:{res_tr.max():.4f} kn")
    print(f"  Clipped {n_clip} rows at ±{CLIP_SIG}σ")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    print(f"  Device: {device}")

    def make_loader(X, y, shuffle=True):
        ds = TensorDataset(
            torch.from_numpy(X).float(),
            torch.from_numpy(y.reshape(-1, 1)).float()
        )
        return DataLoader(ds, batch_size=CFG["batch_size"],
                          shuffle=shuffle,
                          pin_memory=(device.type == "cuda"))

    tr_loader = make_loader(Xtr_s, res_tr_n)
    va_loader = make_loader(Xva_s, res_va_n, shuffle=False)

    mono_n = min(CFG.get("mono_sample_size", 512), len(Xtr_s))
    mi     = np.random.choice(len(Xtr_s), mono_n, replace=False)
    Xm     = torch.from_numpy(Xtr_s[mi]).float().to(device)
    dw_vec = torch.zeros(1, Xtr_s.shape[1], device=device)
    dw_vec[0, 0] = 1.0
    dr_vec = torch.zeros(1, Xtr_s.shape[1], device=device)
    dr_vec[0, 1] = 1.0
    DELTA  = 0.05

    model = PINN(
        in_dim   = Xtr_s.shape[1],
        hidden   = CFG["hidden_dim"],
        n_layers = CFG["num_layers"],
        dropout  = CFG["dropout"],
    ).to(device)

    opt = optim.AdamW(model.parameters(),
                      lr           = CFG["lr"],
                      weight_decay = CFG["weight_decay"])

    warmup = 10
    def lr_fn(ep):
        if ep < warmup:
            return ep / warmup
        p = (ep - warmup) / max(CFG["epochs"] - warmup, 1)
        return 0.5 * (1 + np.cos(np.pi * p)) * (1 - 1e-2) + 1e-2

    sched = optim.lr_scheduler.LambdaLR(opt, lr_fn)

    best_val   = float("inf")
    patience_c = 0
    ema_val    = float("inf")
    tr_hist    = []
    va_hist    = []

    print(f"\n  Training PINN … (batch_size={CFG['batch_size']}, "
          f"mono_sample_size={mono_n}, epochs cap={CFG['epochs']}, "
          f"patience={CFG['patience']})")
    print(f"  {'Epoch':>6}  {'Train MSE':>12}  {'Val MSE':>12}  {'LR':>10}")
    print("  " + "─"*46)

    for ep in range(1, CFG["epochs"] + 1):
        model.train()
        tl = []
        for Xb, rb in tr_loader:
            Xb, rb = Xb.to(device), rb.to(device)
            pred   = model(Xb)
            ld     = nn.functional.mse_loss(pred, rb)

            with torch.no_grad():
                pm = model(Xm)
            dw   = model(Xm + DELTA * dw_vec) - pm.detach()
            dr   = model(Xm + DELTA * dr_vec) - pm.detach()
            loss = ld + CFG["lambda_mono"] * (
                torch.relu(dw) + torch.relu(dr)
            ).mean()

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl.append(ld.item())

        sched.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for Xb, rb in va_loader:
                Xb, rb = Xb.to(device), rb.to(device)
                vl.append(nn.functional.mse_loss(model(Xb), rb).item())

        tr_mse = float(np.mean(tl))
        va_mse = float(np.mean(vl))
        tr_hist.append(tr_mse)
        va_hist.append(va_mse)
        ema_val = va_mse if ep == 1 else 0.8 * ema_val + 0.2 * va_mse

        if ema_val < best_val:
            best_val   = ema_val
            patience_c = 0
            _save_torch(model.state_dict(), PINN_PATH)
        else:
            patience_c += 1

        if patience_c > CFG["patience"]:
            print(f"\n  Early stop ep={ep}  best_val={best_val:.6f}")
            break

        if ep % 25 == 0:
            print(f"  {ep:>6}  {tr_mse:>12.6f}  {va_mse:>12.6f}  "
                  f"{sched.get_last_lr()[0]:>10.2e}")

    model.load_state_dict(
        torch.load(PINN_PATH, map_location=device, weights_only=True)
    )
    model.eval()

    def pinn_pred(X):
        out = []
        for i in range(0, len(X), 4096):
            xb = torch.from_numpy(X[i:i+4096]).float().to(device)
            with torch.no_grad():
                out.append(model(xb).cpu().numpy().flatten())
        return np.concatenate(out) * res_std + res_mean

    pinn_res_tr = pinn_pred(Xtr_s)
    pinn_res_va = pinn_pred(Xva_s)
    pinn_res_te = pinn_pred(Xte_s)

    actual_res_va = yva - Vp_va
    denom = float(np.sum(pinn_res_va ** 2)) + 1e-8
    alpha = float(np.sum(actual_res_va * pinn_res_va) / denom)
    alpha = float(np.clip(alpha, 0.0, 1.2))
    print(f"\n  Residual shrinkage (fit on VAL only): alpha={alpha:.4f}"
          f"  (1.0=full residual, 0.0=fall back to physics)")

    tr_pinn = np.clip(Vp_tr + alpha * pinn_res_tr, 0.0, MAX_SPEED_KN)
    va_pinn = np.clip(Vp_va + alpha * pinn_res_va, 0.0, MAX_SPEED_KN)
    te_pinn = np.clip(Vp_te + alpha * pinn_res_te, 0.0, MAX_SPEED_KN)

    print("\n  Training XGBoost …")

    def xgb_features(Xs, vp, pp):
        return np.column_stack([Xs, vp, pp]).astype(np.float32)

    dtrain = xgb.DMatrix(xgb_features(Xtr_s, Vp_tr, tr_pinn), label=ytr)
    dval   = xgb.DMatrix(xgb_features(Xva_s, Vp_va, va_pinn), label=yva)
    dte    = xgb.DMatrix(xgb_features(Xte_s, Vp_te, te_pinn), label=yte)

    params = {
        "max_depth"        : CFG["xgb_max_depth"],
        "learning_rate"    : CFG["xgb_lr"],
        "subsample"        : CFG["xgb_subsample"],
        "colsample_bytree" : CFG["xgb_colsample"],
        "min_child_weight" : CFG["xgb_min_child"],
        "reg_alpha"        : CFG["xgb_reg_alpha"],
        "reg_lambda"       : CFG["xgb_reg_lambda"],
        "eval_metric"      : "rmse",
        "tree_method"      : "hist",
        "device"           : "cuda" if torch.cuda.is_available() else "cpu",
        "seed"             : 42,
        "verbosity"        : 0,
    }

    xgb_m = xgb.train(
        params,
        dtrain,
        CFG["xgb_n_estimators"],
        [(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=CFG["xgb_early_stopping"],
        verbose_eval=False,
    )
    xgb_m.save_model(XGB_PATH)
    print(f"  XGBoost saved — best iteration: {xgb_m.best_iteration}")

    _save_joblib({
        "res_mean"           : res_mean,
        "res_std"            : res_std,
        "res_shrink_alpha"   : alpha,
        "feature_cols"       : FEATURE_COLS,
        "target"             : TARGET,
        "forecast_horizon_s" : horizon_s,
        "resample_seconds"   : resample_s,
        "n_features_pinn"    : int(Xtr_s.shape[1]),
        "n_features_xgb"     : 15,
        "max_speed_kn"       : MAX_SPEED_KN,
        "units"              : "knots and metres throughout",
        "wave_unit_note"     : "WAVE_HS_M always in METRES",
        "physics_has_lag1"   : True,
        "physics_n_coef"     : 7,
        "data_files"         : [f["path"] for f in CFG["data_files"]],
    }, META_PATH)
    print(f"  meta.pkl saved (res_mean={res_mean:.4f}  res_std={res_std:.4f}"
          f"  alpha={alpha:.4f}  horizon={horizon_s}s  target_grid={resample_s}s)")

    ytr_xg = xgb_m.predict(dtrain)
    yva_xg = xgb_m.predict(dval)
    yte_xg = xgb_m.predict(dte)

    persist_tr = tr_df["SOG_KN"].values.astype(np.float32)
    persist_te = te_df["SOG_KN"].values.astype(np.float32)

    ph_te = metrics_kn(yte, Vp_te)
    pn_te = metrics_kn(yte, te_pinn)
    xg_tr = metrics_kn(ytr, ytr_xg)
    xg_va = metrics_kn(yva, yva_xg)
    xg_te = metrics_kn(yte, yte_xg)
    ps_te = metrics_kn(yte, persist_te)

    W = 90
    print(); print("═"*W)
    print(f"  PINN+XGBoost RESULTS — KNOTS ({mode_label})".center(W))
    print(f"  Test set: {'HELD-OUT FILE(S)' if has_file_test else 'chronological tail slice'}".center(W))
    print("═"*W)
    print(f"  {'Split':<8} {'RMSE(kn)':>9} {'MAE(kn)':>9} {'R²':>7}"
          f" {'±0.5kn':>8} {'±1kn':>7} {'±2kn':>7} {'±5kn':>7}")
    print("  " + "─"*(W-2))
    for sp, m in [("TRAIN", xg_tr), ("VAL", xg_va), ("TEST", xg_te)]:
        print(f"  {sp:<8} {m['rmse_kn']:>9.4f} {m['mae_kn']:>9.4f}"
              f" {m['r2']:>7.4f} {m['p0_5_kn']:>7.1f}%"
              f" {m['p1_kn']:>6.1f}% {m['p2_kn']:>6.1f}% {m['p5_kn']:>6.1f}%")
    if horizon_s > 0:
        print("  " + "─"*(W-2))
        print(f"  {'PERSIST':<8} {ps_te['rmse_kn']:>9.4f} {ps_te['mae_kn']:>9.4f}"
              f" {ps_te['r2']:>7.4f} {ps_te['p0_5_kn']:>7.1f}%"
              f" {ps_te['p1_kn']:>6.1f}% {ps_te['p2_kn']:>6.1f}% {ps_te['p5_kn']:>6.1f}%"
              f"   ← naive baseline (test)")

    print(); print("═"*W)
    print(f"  TEST summary ({mode_label}):")
    print(f"    Knots  →  RMSE={xg_te['rmse_kn']:.4f}  MAE={xg_te['mae_kn']:.4f}"
          f"  ±1kn={xg_te['p1_kn']:.1f}%  ±2kn={xg_te['p2_kn']:.1f}%")
    print(f"    km/h   →  RMSE={xg_te['rmse_kmh']:.3f}  MAE={xg_te['mae_kmh']:.3f}")
    if horizon_s > 0:
        improvement = (ps_te["rmse_kn"] - xg_te["rmse_kn"]) / ps_te["rmse_kn"] * 100
        print(f"    vs persistence baseline: RMSE {ps_te['rmse_kn']:.4f}kn → "
              f"{xg_te['rmse_kn']:.4f}kn  ({improvement:+.1f}% {'better' if improvement > 0 else 'worse'})")
    print(f"  Best XGB iteration: {xgb_m.best_iteration}")
    print("═"*W)

    print(); print("═"*W)
    print("  DIAGNOSTICS".center(W))
    print("═"*W)
    checks = []

    def chk(name, passed, detail=""):
        checks.append(passed)
        print(f"  {'✅' if passed else '❌'} {name}")
        if detail:
            print(f"      {detail}")

    chk("Physics R²>0 on test",      ph_te["r2"] > 0,           f"Physics R²={ph_te['r2']:.4f}")
    chk("PINN improves over physics", pn_te["r2"] > ph_te["r2"], f"Physics R²={ph_te['r2']:.4f} → PINN R²={pn_te['r2']:.4f}")
    chk("XGBoost improves over PINN", xg_te["r2"] > pn_te["r2"], f"PINN R²={pn_te['r2']:.4f} → XGB R²={xg_te['r2']:.4f}")
    chk("Test R²≥0.85",               xg_te["r2"] >= 0.85,       f"Test R²={xg_te['r2']:.4f}")
    chk("RMSE <1.5 kn on test",       xg_te["rmse_kn"] < 1.5,    f"RMSE={xg_te['rmse_kn']:.4f} kn")
    chk("MAE <1.0 kn on test",        xg_te["mae_kn"] < 1.0,     f"MAE={xg_te['mae_kn']:.4f} kn")
    chk("≥90% within ±1 kn",          xg_te["p1_kn"] >= 90.0,    f"{xg_te['p1_kn']:.1f}%")
    chk("≥99% within ±2 kn",          xg_te["p2_kn"] >= 99.0,    f"{xg_te['p2_kn']:.1f}%")

    if horizon_s > 0:
        chk("Beats persistence baseline (RMSE)",
            xg_te["rmse_kn"] < ps_te["rmse_kn"],
            f"XGB={xg_te['rmse_kn']:.4f}kn  vs  Persistence={ps_te['rmse_kn']:.4f}kn")
        chk("Beats persistence baseline (R²)",
            xg_te["r2"] > ps_te["r2"],
            f"XGB R²={xg_te['r2']:.4f}  vs  Persistence R²={ps_te['r2']:.4f}")
        chk("Beats persistence baseline (±0.5kn) — TIGHT bucket",
            xg_te["p0_5_kn"] >= ps_te["p0_5_kn"],
            f"XGB={xg_te['p0_5_kn']:.1f}%  vs  Persistence={ps_te['p0_5_kn']:.1f}% "
            f"— if this fails, the model is smarter on average but less precise "
            f"than 'just guess current speed' at the tightest tolerance")

    ov  = va_hist[-1] / (tr_hist[-1] + 1e-9)
    chk("PINN val/train ratio <5",    ov < 5.0,  f"val/train={ov:.2f}")
    gap_pct = xg_tr["p1_kn"] - xg_te["p1_kn"]
    chk("Train–test ±1kn gap <15%",   gap_pct < 15.0, f"train={xg_tr['p1_kn']:.1f}%  test={xg_te['p1_kn']:.1f}%  gap={gap_pct:.1f}%")

    if has_file_test:
        chk("Test set is a genuinely held-out file (not a ratio slice)",
            True,
            f"held out: {[os.path.basename(f['path']) for f in CFG['data_files'] if f.get('split') == 'test']}")

    passed = sum(checks)
    total  = len(checks)
    score  = passed / total * 100
    print()
    print(f"  Score: {passed}/{total} ({score:.0f}%)")
    if   passed == total: print("  🟢 EXCELLENT")
    elif score  >= 80:    print("  🟡 GOOD — minor issues")
    elif score  >= 60:    print("  🟠 FAIR — review failed checks")
    else:                 print("  🔴 POOR — significant issues")
    if horizon_s > 0 and xg_te["rmse_kn"] >= ps_te["rmse_kn"]:
        print("  ⚠  Model does NOT beat the naive persistence baseline at this")
        print("     horizon — consider shortening forecast_horizon_s, adding")
        print("     forward-looking wind/wave inputs, or treating this horizon")
        print("     as future work rather than a validated result.")
    print(f"  Artefacts saved to: {CFG['checkpoint_dir']}/")
    print("═"*W)

    med_lag1 = float(np.median(l1te))
    print()
    print("═"*W)
    print(f"  INFERENCE EXAMPLES — {mode_label}".center(W))
    print("═"*W)
    print(f"  (using median SOG_LAG1 = {med_lag1:.2f} kn from test set)")
    print()

    def predict_single(wind_app_kn, wave_m, lag1_kn):
        true_wind = max(0.0, wind_app_kn - lag1_kn)
        feat = np.array([[
            true_wind, wave_m,
            lag1_kn, lag1_kn, lag1_kn,
            lag1_kn, 0.0,
            0.0, 0.0,
            0.0, 1.0, 0.0, 1.0
        ]], dtype=np.float32)
        feat_s = scaler.transform(feat)
        vp = physics_predict(
            np.array([true_wind], np.float32),
            np.array([wave_m],    np.float32),
            np.array([lag1_kn],   np.float32),
            coef
        )[0]
        t = torch.from_numpy(feat_s).float().to(device)
        with torch.no_grad():
            res = model(t).item() * res_std + res_mean
        pp   = float(np.clip(vp + alpha * res, 0.0, MAX_SPEED_KN))
        xgf  = np.array([[*feat_s[0], vp, pp]], dtype=np.float32)
        pred = float(xgb_m.predict(xgb.DMatrix(xgf))[0])
        return pred, pred * KN_TO_KMH

    cases = [
        (5,  0.02, "light wind, tiny hump (2 cm)"),
        (10, 0.05, "moderate wind, medium hump (5 cm)"),
        (20, 0.08, "strong wind, large hump (8 cm)"),
        (30, 0.10, "gale, max hump (10 cm)"),
        (40, 0.10, "storm conditions, max hump"),
    ]
    horizon_note = f" (+{horizon_s}s ahead)" if horizon_s > 0 else " (now)"
    print(f"  {'Wind(kn)':>9}  {'Wave(m)':>8}  {'Pred(kn)'+horizon_note:>20}  {'Pred(km/h)':>10}  Conditions")
    print("  " + "─"*80)
    for w, wv, desc in cases:
        pkn, pkm = predict_single(w, wv, med_lag1)
        print(f"  {w:>9.0f}  {wv:>8.3f}  {pkn:>20.3f}  {pkm:>10.2f}  {desc}")
    print("═"*W)

    print("\n  Generating plots …")
    N  = min(2000, len(yte))
    ix = np.random.choice(len(yte), N, replace=False)
    lo = max(0.0, float(yte.min()) - 1)
    hi = float(yte.max()) + 1

    fig = plt.figure(figsize=(14, 12))
    fig.suptitle(
        f"Grey-Box Vessel Model v7.5 — PINN+XGBoost — {mode_label}\n"
        f"Test set: {'held-out file(s)' if has_file_test else 'chronological tail slice'} | "
        f"{resample_s}s target grid (per-file native cadence preserved)",
        fontsize=12, fontweight="bold", y=0.98
    )
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

    ax_sc = fig.add_subplot(gs[0, 0])
    ax_sc.scatter(yte[ix], yte_xg[ix], alpha=0.25, s=8,
                  color=COLORS["xgb"], linewidths=0)
    ax_sc.plot([lo, hi], [lo, hi], "--", color=COLORS["ideal"], lw=1.2)
    ax_sc.set_title(
        f"Scatter — TEST\nR²={xg_te['r2']:.4f}  "
        f"RMSE={xg_te['rmse_kn']:.3f}kn  MAE={xg_te['mae_kn']:.3f}kn",
        fontsize=7.5
    )
    ax_sc.set_xlabel(f"Actual{horizon_note} (kn)", fontsize=7)
    ax_sc.set_ylabel(f"Predicted{horizon_note} (kn)", fontsize=7)
    ax_sc.set_xlim(lo, hi); ax_sc.set_ylim(lo, hi)
    ax_sc.tick_params(labelsize=6); ax_sc.grid(True, alpha=0.3)
    ax2 = ax_sc.twinx()
    ax2.set_ylim(lo * KN_TO_KMH, hi * KN_TO_KMH)
    ax2.set_ylabel("(km/h)", fontsize=6, color="grey")
    ax2.tick_params(labelsize=5, colors="grey")

    ax_ts = fig.add_subplot(gs[0, 1:])
    n_ts  = min(600, len(yte))
    x_ts  = np.arange(n_ts)
    ax_ts.plot(x_ts, yte[:n_ts],    color="black",       lw=0.8, label="Actual",   alpha=0.85)
    ax_ts.plot(x_ts, yte_xg[:n_ts], color=COLORS["xgb"], lw=0.8, label="PINN+XGB", alpha=0.85)
    if horizon_s > 0:
        ax_ts.plot(x_ts, persist_te[:n_ts], color=COLORS["persist"], lw=0.8,
                   ls=":", label="Persistence", alpha=0.7)
    ax2_ts = ax_ts.twinx()
    ax2_ts.set_ylim(ax_ts.get_ylim()[0]*KN_TO_KMH, ax_ts.get_ylim()[1]*KN_TO_KMH)
    ax2_ts.set_ylabel("km/h", fontsize=6, color="grey")
    ax2_ts.tick_params(labelsize=5, colors="grey")
    ax_ts.set_title(f"Time series — TEST (first 600 samples), {mode_label}", fontsize=8)
    ax_ts.set_xlabel("Sample", fontsize=7)
    ax_ts.set_ylabel("SOG (kn)", fontsize=7)
    ax_ts.legend(fontsize=6); ax_ts.tick_params(labelsize=6)
    ax_ts.grid(True, alpha=0.3)

    errs = yte_xg - yte
    ax_h = fig.add_subplot(gs[1, 0])
    ax_h.hist(errs, bins=80, color=COLORS["xgb"], alpha=0.75, edgecolor="none")
    ax_h.axvline(0,    color="black", lw=1.0, ls="--")
    ax_h.axvline( 1.0, color="green", lw=0.8, ls=":", label="±1 kn")
    ax_h.axvline(-1.0, color="green", lw=0.8, ls=":")
    ax_h.set_title(
        f"Error distribution — TEST\n"
        f"mean={errs.mean():.4f} kn ({errs.mean()*KN_TO_KMH:.3f} km/h)",
        fontsize=8
    )
    ax_h.set_xlabel("Error (kn)", fontsize=7)
    ax_h.set_ylabel("Count", fontsize=7)
    ax_h.legend(fontsize=6); ax_h.tick_params(labelsize=6)
    ax_h.grid(True, alpha=0.3)

    ax_c  = fig.add_subplot(gs[1, 1])
    ae_c  = np.sort(np.abs(errs))
    cdf   = np.arange(1, len(ae_c) + 1) / len(ae_c)
    ax_c.plot(ae_c, cdf * 100, color=COLORS["xgb"], lw=1.5)
    for v, key, col, ls in [
        (0.5, "p0_5_kn", "purple", ":"),
        (1.0, "p1_kn",   "blue",   ":"),
        (2.0, "p2_kn",   "green",  "--"),
        (5.0, "p5_kn",   "orange", "--"),
    ]:
        pct = xg_te[key]
        ax_c.axvline(v, color=col, ls=ls, lw=0.9, label=f"±{v}kn: {pct:.1f}%")
    ax_c.set_title("CDF — TEST (knots)", fontsize=8)
    ax_c.set_xlabel("Abs error (kn)", fontsize=7)
    ax_c.set_ylabel("% within", fontsize=7)
    ax_c.legend(fontsize=5.5); ax_c.set_xlim(0, 8)
    ax_c.set_ylim(0, 100); ax_c.tick_params(labelsize=6)
    ax_c.grid(True, alpha=0.3)

    ax_b   = fig.add_subplot(gs[1, 2])
    if horizon_s > 0:
        splits = ["Persist", "Physics", "PINN", "XGB"]
        v_kn   = [ps_te["rmse_kn"], ph_te["rmse_kn"], pn_te["rmse_kn"], xg_te["rmse_kn"]]
        bar_colors = [COLORS["persist"], COLORS["physics"], COLORS["pinn"], COLORS["xgb"]]
        ax_b.set_title("TEST RMSE by stage\n(forecast vs naive baseline)", fontsize=8)
        x_b = np.arange(len(splits))
        bars = ax_b.bar(x_b, v_kn, color=bar_colors, edgecolor="none")
        for bar, v in zip(bars, v_kn):
            ax_b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                      f"{v:.3f}", ha="center", va="bottom", fontsize=6)
        ax_b.set_xticks(x_b); ax_b.set_xticklabels(splits, fontsize=7)
        ax_b.set_ylabel("RMSE (kn)", fontsize=7)
        ax_b.tick_params(labelsize=6); ax_b.grid(True, alpha=0.3, axis="y")
        ax_b.set_ylim(0, max(v_kn) * 1.35)
    else:
        splits = ["Train", "Val", "Test"]
        v_kn   = [xg_tr["rmse_kn"],  xg_va["rmse_kn"],  xg_te["rmse_kn"]]
        v_km   = [xg_tr["rmse_kmh"], xg_va["rmse_kmh"], xg_te["rmse_kmh"]]
        x_b    = np.arange(3); w = 0.35
        b1 = ax_b.bar(x_b - w/2, v_kn, w, color=COLORS["xgb"],  label="knots", edgecolor="none")
        b2 = ax_b.bar(x_b + w/2, v_km, w, color=COLORS["xgb"],  label="km/h",  edgecolor="none", alpha=0.5)
        for bar, v in zip(b1, v_kn):
            ax_b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                      f"{v:.3f}", ha="center", va="bottom", fontsize=6)
        for bar, v in zip(b2, v_km):
            ax_b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                      f"{v:.3f}", ha="center", va="bottom", fontsize=6)
        ax_b.set_title("RMSE by split\nSolid=kn  Faded=km/h", fontsize=8)
        ax_b.set_xticks(x_b); ax_b.set_xticklabels(splits, fontsize=7)
        ax_b.set_ylabel("RMSE", fontsize=7); ax_b.legend(fontsize=6)
        ax_b.tick_params(labelsize=6); ax_b.grid(True, alpha=0.3, axis="y")
        ax_b.set_ylim(0, max(v_km) * 1.35)

    plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✅ Plot saved → {PLOT_PATH}")
    print(f"  MODEL_DIR for inference = '{CFG['checkpoint_dir']}'")
    print()
    print("═"*W)


if __name__ == "__main__":
    train()